In [1]:
import numpy as np
from dataclasses import dataclass

import plotly.graph_objects as go

In [2]:
@dataclass
class Node3D:
    id :int
    x: float
    y: float
    z: float
    theta_x: float = 0.0
    theta_y: float = 0.0
    theta_z: float = 0.0

@dataclass
class Beam3D:
    id: int
    node_i: Node3D
    node_j: Node3D

    # 断面積
    A: float = 1.0

    # 断面二次モーメント
    I: float = 1.0

    # 縦弾性係数(ヤング率)
    E: float = 1.0

    # 横弾性係数
    G: float = 1.0

    # ねじり定数
    J: float = 1.0

    @property
    def length(self):
        i, j = self.node_i, self.node_j
        v = np.array([j.x-i.x, j.y-i.y, j.z-i.z])
        return float(np.linalg.norm(v))

    # X軸(引張/圧縮)
    def axial_matrix(self):
        L = self.length
        A, E = self.A, self.E
        factor = A * E / L
        M = np.array([
            [1, -1],
            [-1, 1],
        ])
        return factor * M

    # X軸(ねじれ)
    def tortion_matrix(self):
        L = self.length
        G, J = self.G, self.J
        factor = G * J / L
        M = np.array([
            [1, -1],
            [-1, 1],
        ])
        return factor * M

    # Y,Z軸(曲げ)
    def bend_matrix(self):
        L = self.length
        I, E = self.I, self.E
        factor = E * I / (L**3)
        M = np.array([
            [12,  6*L,    -12,  6*L],
            [6*L, 4*L**2, -6*L, 2*L**2],
            [-12, -6*L,   12,   -6*L],
            [6*L, 2*L**2, -6*L, 4*L**2],
        ])
        return factor * M

In [3]:
# 要素
n0 = Node3D(0, 0.0, 0.0, 0.0)
n1 = Node3D(1, 1.0, 0.0, 0.0)
n2 = Node3D(2, 0.0, 1.0, 0.0)
n3 = Node3D(3, 1.0, 1.0, 0.0)
n4 = Node3D(4, 0.0, 0.0, 1.0)
n5 = Node3D(5, 1.0, 0.0, 1.0)
n6 = Node3D(6, 0.0, 1.0, 1.0)
n7 = Node3D(7, 1.0, 1.0, 1.0)

nodes = [n0, n1, n2, n3, n4, n5, n6, n7]

# 梁
b0 = Beam3D(0, n0, n1)
b1 = Beam3D(1, n2, n3)
b2 = Beam3D(2, n4, n5)
b3 = Beam3D(3, n6, n7)

b4 = Beam3D(4, n0, n2)
b5 = Beam3D(5, n1, n3)
b6 = Beam3D(6, n4, n6)
b7 = Beam3D(7, n5, n7)

b8 = Beam3D(8, n0, n4)
b9 = Beam3D(9, n1, n5)
b10 = Beam3D(10, n2, n6)
b11 = Beam3D(11, n3, n7)

beams = [b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10, b11]

In [7]:
fig = go.Figure()

for n in nodes:
    x, y, z = n.x, n.y, n.z
    text = F"n{n.id}"
    fig.add_traces(data=[go.Scatter3d(x=[x], y=[y], z=[z], text=[text], mode='markers+text', marker={"size": 3.0, "color": "black"}, showlegend=False)])

for b in beams:
    i, j = b.node_i, b.node_j
    x = [i.x, j.x]
    y = [i.y, j.y]
    z = [i.z, j.z]

    Mx = (i.x + j.x) / 2
    My = (i.y + j.y) / 2
    Mz = (i.z + j.z) / 2
    text = F"b{b.id}"
    fig.add_traces(data=[go.Scatter3d(x=x, y=y, z=z, mode='lines+text', line={"width": 1.0, "color": "black"},showlegend=False)])
    fig.add_traces(data=[go.Scatter3d(x=[Mx], y=[My], z=[Mz], text=[text], mode='text',showlegend=False)])

fig.update_layout(margin=dict(l=0, r=0, b=0, t=0))

fig.show()